In [1]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import tensorflow as tf
from collections import deque

In [2]:
# Load trained model and normalization files
model = tf.keras.models.load_model("best_tcn_gesture_model.keras")

mean = np.load("landmark_mean.npy")
std = np.load("landmark_std.npy")

print("Model loaded!")
print("Mean shape:", mean.shape)
print("Std shape:", std.shape)

Model loaded!
Mean shape: (1, 1, 63)
Std shape: (1, 1, 63)


In [3]:
# MediaPipe Tasks API setup
base_options = python.BaseOptions(model_asset_path="hand_landmarker.task")

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5
)

detector = vision.HandLandmarker.create_from_options(options)

I0000 00:00:1780212186.709546 65399039 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
I0000 00:00:1780212186.809089 65399039 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1780212186.827094 65399042 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780212186.843011 65399045 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [4]:
# Constants
SEQUENCE_LENGTH = 37
NUM_LANDMARKS = 21
FEATURES_PER_FRAME = NUM_LANDMARKS * 3  # 63

# Rolling buffer stores the most recent 37 frames of landmarks
landmark_buffer = deque(maxlen=SEQUENCE_LENGTH)

In [5]:
def extract_landmarks_from_image(rgb_image):
    """
    Takes one RGB frame from webcam.
    Returns:
    - landmarks: shape (63,)
    - detection_result: MediaPipe detection result for drawing/debugging
    """

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_image
    )

    detection_result = detector.detect(mp_image)

    if detection_result.hand_landmarks:
        hand_landmarks = detection_result.hand_landmarks[0]

        landmarks = []
        for lm in hand_landmarks:
            landmarks.extend([lm.x, lm.y, lm.z])

        return np.array(landmarks, dtype=np.float32), detection_result

    else:
        return np.zeros(FEATURES_PER_FRAME, dtype=np.float32), detection_result

In [6]:
def predict_gesture(landmark_sequence):
    """
    landmark_sequence shape: (37, 63)
    Returns predicted class index and confidence.
    """

    x = landmark_sequence.astype("float32")

    mean_fixed = np.squeeze(mean)
    std_fixed = np.squeeze(std)

    # Normalize same way as training
    x = (x - mean_fixed) / std_fixed

    # Add batch dimension: (1, 37, 63)
    x = np.expand_dims(x, axis=0)

    probs = model.predict(x, verbose=0)

    class_idx = int(np.argmax(probs, axis=1)[0])
    confidence = float(np.max(probs))

    return class_idx, confidence

In [7]:
'''
# Use 0 or 1 depending on which camera worked better for you
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Try changing VideoCapture(1) to VideoCapture(0).")

latest_prediction = "Waiting..."
latest_confidence = 0.0
frame_count = 0

# Predict every few frames instead of every single frame
predict_every_n_frames = 5

try:
    while True:
        success, frame = cap.read()

        if not success:
            print("Failed to grab frame")
            break

        # Mirror image so it feels natural
        frame = cv2.flip(frame, 1)

        # Convert BGR to RGB for MediaPipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Extract landmarks
        landmarks, detection_result = extract_landmarks_from_image(rgb_frame)

        # Add current frame landmarks to 37-frame rolling buffer
        landmark_buffer.append(landmarks)

        # Draw hand landmark dots manually
        if detection_result.hand_landmarks:
            h, w, _ = frame.shape

            for lm in detection_result.hand_landmarks[0]:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)

        # Predict once buffer is full
        if len(landmark_buffer) == SEQUENCE_LENGTH and frame_count % predict_every_n_frames == 0:
            landmark_sequence = np.array(landmark_buffer)  # shape: (37, 63)

            class_idx, confidence = predict_gesture(landmark_sequence)

            latest_prediction = f"Class {class_idx}"
            latest_confidence = confidence

            print(f"Prediction: {latest_prediction} | Confidence: {latest_confidence:.3f}")

        # Display prediction on screen
        cv2.putText(
            frame,
            f"Prediction: {latest_prediction}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Confidence: {latest_confidence:.2f}",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            "Press q to quit",
            (20, 120),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2
        )

        cv2.imshow("Real-Time Gesture Recognition", frame)

        frame_count += 1

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    detector.close()
    print("Webcam released.")
'''

'\n# Use 0 or 1 depending on which camera worked better for you\ncap = cv2.VideoCapture(1)\n\nif not cap.isOpened():\n    raise RuntimeError("Could not open webcam. Try changing VideoCapture(1) to VideoCapture(0).")\n\nlatest_prediction = "Waiting..."\nlatest_confidence = 0.0\nframe_count = 0\n\n# Predict every few frames instead of every single frame\npredict_every_n_frames = 5\n\ntry:\n    while True:\n        success, frame = cap.read()\n\n        if not success:\n            print("Failed to grab frame")\n            break\n\n        # Mirror image so it feels natural\n        frame = cv2.flip(frame, 1)\n\n        # Convert BGR to RGB for MediaPipe\n        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n\n        # Extract landmarks\n        landmarks, detection_result = extract_landmarks_from_image(rgb_frame)\n\n        # Add current frame landmarks to 37-frame rolling buffer\n        landmark_buffer.append(landmarks)\n\n        # Draw hand landmark dots manually\n        if 

In [10]:
camera_index = 0  # change to 1 if camera 0 does not work
import cv2
import time
import numpy as np

cap = cv2.VideoCapture(camera_index, cv2.CAP_AVFOUNDATION)

# Lower resolution
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 15)

if not cap.isOpened():
    raise RuntimeError(f"Could not open camera {camera_index}")

# Warm up camera
print("Warming up camera...")
for _ in range(20):
    ret, frame = cap.read()
    time.sleep(0.05)

ret, frame = cap.read()
if not ret:
    cap.release()
    cv2.destroyAllWindows()
    raise RuntimeError("Camera opened, but could not read frames. Restart kernel or try camera_index = 1.")

print("Camera is working. Frame shape:", frame.shape)

frame_count = 0
print_every_n_frames = 30

start_time = time.time()
max_seconds = 30  # auto-stops after 30 seconds

try:
    while True:
        success, frame = cap.read()

        if not success:
            print("Failed to grab frame")
            break

        # Mirror image
        frame = cv2.flip(frame, 1)

        # Convert BGR to RGB for MediaPipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Extract landmarks using your newer MediaPipe Tasks function
        landmarks, detection_result = extract_landmarks_from_image(rgb_frame)

        # Add current frame landmarks to 37-frame rolling buffer
        landmark_buffer.append(landmarks)

        # Draw landmark dots if hand is detected
        if detection_result.hand_landmarks:
            h, w, _ = frame.shape

            for lm in detection_result.hand_landmarks[0]:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 4, (0, 255, 0), -1)

            status_text = "Hand detected"
            status_color = (0, 255, 0)
        else:
            status_text = "No hand detected"
            status_color = (0, 0, 255)

        # Confirm buffer shape every 30 frames
        if len(landmark_buffer) == SEQUENCE_LENGTH and frame_count % print_every_n_frames == 0:
            landmark_sequence = np.array(landmark_buffer)

            print("Buffer full!")
            print("landmark_sequence shape:", landmark_sequence.shape)

            detected_frames = np.sum(~np.all(landmark_sequence == 0, axis=1))
            print("Frames with hand detected:", detected_frames, "/", SEQUENCE_LENGTH)

        # Display text on webcam window
        cv2.putText(
            frame,
            status_text,
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            status_color,
            2
        )

        cv2.putText(
            frame,
            f"Buffer: {len(landmark_buffer)}/{SEQUENCE_LENGTH}",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            "Click this window, then press q or ESC to quit",
            (20, 120),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2
        )

        cv2.imshow("MediaPipe Landmark Buffer Test", frame)

        frame_count += 1

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q") or key == 27:
            print("Stopped by user.")
            break

        if time.time() - start_time > max_seconds:
            print("Auto-stopped after 30 seconds.")
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam released.")

Warming up camera...
Camera is working. Frame shape: (480, 640, 3)


W0000 00:00:1780212213.009904 65399048 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
2026-05-31 00:23:34.229 python[45095:65398921] +[IMKClient subclass]: chose IMKClient_Legacy
2026-05-31 00:23:34.229 python[45095:65398921] +[IMKInputSession subclass]: chose IMKInputSession_Legacy


Buffer full!
landmark_sequence shape: (37, 63)
Frames with hand detected: 37 / 37
Buffer full!
landmark_sequence shape: (37, 63)
Frames with hand detected: 29 / 37
Buffer full!
landmark_sequence shape: (37, 63)
Frames with hand detected: 0 / 37
Stopped by user.
Webcam released.
